# 02 - Results: base vs LoRA vs LoRA + LaBSE

Reads the JSON files written by `scripts/evaluate.py` from `results/`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS_DIR = ROOT / "results"

MODEL_ORDER = ["base", "raw", "labse"]
MODEL_LABELS = {"base": "Base Whisper Medium", "raw": "LoRA", "labse": "LoRA + LaBSE"}
MODEL_COLORS = {"base": "#B0B0B0", "raw": "#808080", "labse": "#404040"}

rows = []
for path in sorted(RESULTS_DIR.glob("*.json")):
    r = json.loads(path.read_text(encoding="utf-8"))
    rows.append({"lang": r["lang"], "model": r["model"], "source": "all", **r["overall"]})
    for source, metrics in r["by_source"].items():
        rows.append({"lang": r["lang"], "model": r["model"], "source": source, **metrics})

df = pd.DataFrame(rows)
overall = df[df.source == "all"]
overall.pivot(index="model", columns="lang", values=["wer_norm", "cer_norm", "semantic_similarity"]).reindex(MODEL_ORDER).round(2)

In [ ]:
def plot_metric(metric: str, ylabel: str, title: str, fmt: str = "{:.2f}"):
    langs = sorted(overall.lang.unique())
    fig, axes = plt.subplots(1, len(langs), figsize=(7 * len(langs), 5), squeeze=False)
    for ax, lang in zip(axes[0], langs):
        sub = overall[overall.lang == lang].set_index("model").reindex(MODEL_ORDER).dropna(subset=[metric])
        values = sub[metric].tolist()
        ax.bar([MODEL_LABELS[m] for m in sub.index], values, color=[MODEL_COLORS[m] for m in sub.index], width=0.5)
        for i, value in enumerate(values):
            ax.text(i, value, fmt.format(value), ha="center", va="bottom")
        ax.set_title(f"{title} ({lang})")
        ax.set_ylabel(ylabel)
        ax.set_ylim(0, max(values) * 1.15 if values else 1)
        ax.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()


plot_metric("wer_norm", "Word Error Rate (%)", "WER (normalized, lower is better)", "{:.2f}%")
plot_metric("cer_norm", "Character Error Rate (%)", "CER (normalized, lower is better)", "{:.2f}%")
plot_metric("semantic_similarity", "LaBSE cosine similarity", "Semantic similarity (higher is better)", "{:.3f}")

## Per-source breakdown

In [ ]:
(
    df[df.source != "all"]
    .pivot_table(index=["lang", "source"], columns="model", values="wer_norm")
    .reindex(columns=MODEL_ORDER)
    .round(2)
)